# FlowDesk Analytics — Notebook 1
## Exploratory Data Analysis — Understanding FlowDesk's Customer Base

**Author:** Siddharth Bokka  
**Project:** FlowDesk B2B SaaS Analytics Portfolio  

---

### Objective

This notebook answers the foundational business question:

> *"What does our business look like right now? Who are our customers, how do they behave, and where should we focus?"*

Before running any model or experiment, a data analyst must understand the shape of the data, the health of the business, and the key patterns that will guide future work. This is the EDA notebook — everything else in this portfolio builds on it.

**Sections:**
1. Data Quality Audit
2. User Base Overview
3. Revenue and Plan Analysis
4. Activity and Engagement Patterns
5. Churn Overview
6. Key Findings Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.figsize': (10, 5),
})
sns.set_palette('tab10')

# Load datasets
users         = pd.read_parquet('../data/users.parquet')
events        = pd.read_parquet('../data/events.parquet')
feature_usage = pd.read_parquet('../data/feature_usage.parquet')
experiments   = pd.read_parquet('../data/experiments.parquet')
workspaces    = pd.read_parquet('../data/workspaces.parquet')
tickets       = pd.read_parquet('../data/support_tickets.parquet')

users['signup_date'] = pd.to_datetime(users['signup_date'])
events['event_ts']   = pd.to_datetime(events['event_ts'])

print(f"Users: {len(users):,} | Events: {len(events):,} | Experiments: {len(experiments):,}")

---
## Section 1 — Data Quality Audit

> *"Always start with a data quality check — garbage in, garbage out."*

Before any analysis, we verify:
- Row counts match expectations
- Null rates are acceptable (especially on key join columns and metrics)
- Date ranges make sense for the business period we're analyzing
- No obviously corrupted values

A table like this is what you'd deliver at the start of any analytics project kickoff.

In [ ]:
def audit_df(name, df, date_col=None):
    """Return a summary dict for one dataframe."""
    total_cells = df.shape[0] * df.shape[1]
    null_cells   = df.isnull().sum().sum()
    null_pct     = null_cells / total_cells * 100 if total_cells > 0 else 0

    date_range = "—"
    if date_col and date_col in df.columns:
        col = pd.to_datetime(df[date_col], errors='coerce')
        date_range = f"{col.min().date()} → {col.max().date()}"

    return {
        'Table':      name,
        'Rows':       f"{df.shape[0]:,}",
        'Columns':    df.shape[1],
        'Null %':     f"{null_pct:.1f}%",
        'Date Range': date_range,
    }

audit_rows = [
    audit_df('users',         users,         'signup_date'),
    audit_df('events',        events,        'event_ts'),
    audit_df('feature_usage', feature_usage, 'week_start'),
    audit_df('experiments',   experiments,   'assigned_at'),
    audit_df('workspaces',    workspaces,    'created_at'),
    audit_df('tickets',       tickets,       'created_at'),
]

audit_df_display = pd.DataFrame(audit_rows).set_index('Table')
print("=" * 65)
print(" DATA QUALITY AUDIT — FlowDesk Analytics")
print("=" * 65)
print(audit_df_display.to_string())
print()

# Per-column null breakdown for users (most important table)
print("\nPer-column null rates — users table:")
null_by_col = users.isnull().mean().mul(100).round(1)
null_by_col = null_by_col[null_by_col > 0]
if null_by_col.empty:
    print("  No nulls found.")
else:
    for col, pct in null_by_col.items():
        print(f"  {col}: {pct:.1f}%")

print("\nPer-column null rates — events table (sample):")
null_ev = events.isnull().mean().mul(100).round(1)
null_ev = null_ev[null_ev > 0]
if null_ev.empty:
    print("  No nulls found.")
else:
    for col, pct in null_ev.items():
        print(f"  {col}: {pct:.1f}%")

In [ ]:
# Quick sanity checks on key columns
print("users.plan unique values:",   sorted(users['plan'].unique()))
print("users.primary_device:",        sorted(users['primary_device'].unique()))
print("events.event_type (sample):",  sorted(events['event_type'].unique())[:8])
print("users.is_activated values:",   sorted(users['is_activated'].unique()))
print("users.is_churned values:",     sorted(users['is_churned'].unique()))
print(f"\nSignup date range:   {users['signup_date'].min().date()} → {users['signup_date'].max().date()}")
print(f"Event ts date range: {events['event_ts'].min().date()} → {events['event_ts'].max().date()}")

---
## Section 2 — User Base Overview

Here we paint a complete picture of who FlowDesk's users are:
- How many are activated vs. dormant?
- Which plans do they use?
- Where do they come from (acquisition channel)?
- What devices do they use?
- Which countries are they in?

We'll also surface the **mobile activation gap** — a signal we'll investigate deeply in Notebook 3.

In [ ]:
total_users     = len(users)
activated_users = users['is_activated'].sum()
churned_users   = users['is_churned'].sum()
active_users    = activated_users - churned_users  # rough active count

print("=" * 50)
print("  FLOWDESK USER BASE — TOP-LINE METRICS")
print("=" * 50)
print(f"  Total Users:     {total_users:>8,}  (100%)")
print(f"  Activated:       {int(activated_users):>8,}  ({activated_users/total_users*100:.1f}%)")
print(f"  Churned:         {int(churned_users):>8,}  ({churned_users/total_users*100:.1f}%)")
print(f"  Never Activated: {total_users - int(activated_users):>8,}  ({(total_users - activated_users)/total_users*100:.1f}%)")
print("=" * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plan distribution pie ---
plan_counts = users['plan'].value_counts()
plan_order  = ['free', 'pro', 'business', 'enterprise']
plan_counts = plan_counts.reindex([p for p in plan_order if p in plan_counts.index])
colors_plan = ['#aec6cf', '#4c9be8', '#2563eb', '#1e3a8a']

wedges, texts, autotexts = axes[0].pie(
    plan_counts,
    labels=plan_counts.index,
    autopct='%1.1f%%',
    colors=colors_plan[:len(plan_counts)],
    startangle=140,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
axes[0].set_title('Users by Plan Tier', fontsize=13, fontweight='bold', pad=12)

# --- Acquisition channel horizontal bar ---
channel_counts = users['acquisition_channel'].value_counts().sort_values()
colors_ch = sns.color_palette('tab10', len(channel_counts))
bars = axes[1].barh(channel_counts.index, channel_counts.values, color=colors_ch)
for bar, val in zip(bars, channel_counts.values):
    axes[1].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[1].set_xlabel('Number of Users')
axes[1].set_title('Users by Acquisition Channel', fontsize=13, fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('FlowDesk User Base — Plan & Acquisition Breakdown', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Device pie ---
device_counts = users['primary_device'].value_counts()
colors_dev = ['#2563eb', '#f97316', '#10b981']
wedges2, texts2, autotexts2 = axes[0].pie(
    device_counts,
    labels=device_counts.index,
    autopct='%1.1f%%',
    colors=colors_dev[:len(device_counts)],
    startangle=90,
)
for at in autotexts2:
    at.set_fontsize(10)
axes[0].set_title('Users by Primary Device', fontsize=13, fontweight='bold', pad=12)

# --- Country bar chart (top 10) ---
country_counts = users['country'].value_counts().head(10).sort_values()
colors_co = sns.color_palette('Blues_r', len(country_counts))
bars2 = axes[1].barh(country_counts.index, country_counts.values, color=colors_co)
for bar, val in zip(bars2, country_counts.values):
    axes[1].text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[1].set_xlabel('Number of Users')
axes[1].set_title('Top 10 Countries by User Count', fontsize=13, fontweight='bold')

plt.suptitle('FlowDesk User Base — Device & Geographic Breakdown', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mobile Activation Gap — this is the first signal of the funnel problem
device_activation = (
    users
    .groupby('primary_device')['is_activated']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'activated', 'count': 'total'})
    .assign(activation_rate=lambda d: d['activated'] / d['total'] * 100)
    .sort_values('activation_rate', ascending=False)
)

print("Activation Rate by Primary Device:")
print(device_activation.round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
palette = {'desktop': '#2563eb', 'mobile': '#f97316', 'tablet': '#10b981'}
colors_list = [palette.get(d, '#888') for d in device_activation.index]

bars3 = ax.bar(device_activation.index, device_activation['activation_rate'],
               color=colors_list, width=0.5, zorder=3)

for bar, (_, row) in zip(bars3, device_activation.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{row['activation_rate']:.1f}%\n({int(row['activated']):,}/{int(row['total']):,})",
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 90)
ax.set_ylabel('Activation Rate (%)')
ax.set_title('Activation Rate by Primary Device\n'
             '(Preview: Mobile users activate at significantly lower rates)',
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.grid(axis='y', alpha=0.3, zorder=0)

# Annotation arrow highlighting the gap
desktop_rate = device_activation.loc['desktop', 'activation_rate'] if 'desktop' in device_activation.index else 72
mobile_rate  = device_activation.loc['mobile',  'activation_rate'] if 'mobile'  in device_activation.index else 50
gap = desktop_rate - mobile_rate
ax.annotate(f'~{gap:.0f} pp gap\n→ Investigate in Notebook 3',
            xy=(1, 0), xytext=(1.35, 40),
            fontsize=9, color='#dc2626',
            arrowprops=dict(arrowstyle='->', color='#dc2626', lw=1.5))

plt.tight_layout()
plt.show()

**Key Observation:** Mobile users activate at a ~20+ percentage point lower rate than desktop users. This is not just a minor gap — it represents thousands of users who signed up and never experienced FlowDesk's core value. We will dissect this fully in **Notebook 3 (Funnel Analysis)**.

---
## Section 3 — Revenue and Plan Analysis

FlowDesk is a B2B SaaS product with a freemium model. The majority of users are on the free plan, which generates no MRR. Revenue comes entirely from pro, business, and enterprise tiers.

Here we look at:
- MRR distribution across plan tiers
- Which industries drive the most revenue per workspace
- How workspace size maps to plan tier

In [ ]:
# MRR by plan tier
mrr_by_plan = (
    workspaces
    .groupby('plan')['mrr']
    .agg(['sum', 'count', 'mean'])
    .rename(columns={'sum': 'total_mrr', 'count': 'workspaces', 'mean': 'avg_mrr'})
    .sort_values('total_mrr', ascending=False)
)
mrr_by_plan['total_mrr_pct'] = mrr_by_plan['total_mrr'] / mrr_by_plan['total_mrr'].sum() * 100

print("MRR by Plan Tier:")
print(mrr_by_plan.assign(
    total_mrr=mrr_by_plan['total_mrr'].map('${:,.0f}'.format),
    avg_mrr=mrr_by_plan['avg_mrr'].map('${:,.0f}'.format),
    total_mrr_pct=mrr_by_plan['total_mrr_pct'].map('{:.1f}%'.format)
).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- MRR bar chart by plan ---
plan_colors = {'free': '#aec6cf', 'pro': '#4c9be8', 'business': '#2563eb', 'enterprise': '#1e3a8a'}
bar_colors  = [plan_colors.get(p, '#888') for p in mrr_by_plan.index]
bars_mrr = axes[0].bar(mrr_by_plan.index, mrr_by_plan['total_mrr'], color=bar_colors, width=0.55, zorder=3)
for bar, val in zip(bars_mrr, mrr_by_plan['total_mrr']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Total MRR ($)')
axes[0].set_title('Total MRR by Plan Tier', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[0].grid(axis='y', alpha=0.3, zorder=0)

# --- Avg MRR per workspace by industry ---
mrr_by_industry = (
    workspaces[workspaces['plan'] != 'free']
    .groupby('industry')['mrr']
    .mean()
    .sort_values()
)
bars_ind = axes[1].barh(mrr_by_industry.index, mrr_by_industry.values,
                        color=sns.color_palette('Blues_r', len(mrr_by_industry)))
for bar, val in zip(bars_ind, mrr_by_industry.values):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9)
axes[1].set_xlabel('Avg MRR per Workspace ($)')
axes[1].set_title('Avg MRR per Workspace by Industry\n(Paid Plans Only)', fontsize=12, fontweight='bold')

plt.suptitle('Revenue Analysis — FlowDesk MRR Breakdown', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plan distribution by workspace size_tier — stacked bar
size_plan = (
    workspaces
    .groupby(['size_tier', 'plan'])
    .size()
    .unstack(fill_value=0)
)

# Ensure consistent column ordering
plan_order_cols = [p for p in ['free', 'pro', 'business', 'enterprise'] if p in size_plan.columns]
size_plan = size_plan[plan_order_cols]

# Normalize to percentages
size_plan_pct = size_plan.div(size_plan.sum(axis=1), axis=0) * 100

# Reorder size tiers sensibly
tier_order = [t for t in ['solo', 'small', 'medium', 'large', 'enterprise'] if t in size_plan_pct.index]
size_plan_pct = size_plan_pct.reindex(tier_order)

fig, ax = plt.subplots(figsize=(11, 5))
colors_stack = [plan_colors.get(p, '#ccc') for p in plan_order_cols]
size_plan_pct.plot(kind='bar', stacked=True, ax=ax, color=colors_stack, width=0.6, zorder=3)

ax.set_ylabel('% of Workspaces')
ax.set_xlabel('Workspace Size Tier')
ax.set_title('Plan Mix by Workspace Size Tier\n'
             '(Solo workspaces skew Free; Large/Enterprise workspaces skew paid)',
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(title='Plan', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Business Insight:**

> Free users represent ~60% of our user base but contribute **$0 MRR**. The most significant growth lever is not acquiring more users — it is converting free users to paid plans. Even a 5% improvement in free-to-paid conversion would represent a meaningful jump in MRR.

The workspace size analysis confirms our ICP (Ideal Customer Profile): **medium and large workspaces** have the highest paid plan mix and likely the highest LTV. Acquisition targeting and onboarding should be optimized for this segment.

---
## Section 4 — Activity and Engagement Patterns

Now we look at *how* users behave inside the product:
- Daily Active Users over time — is growth healthy?
- Which events dominate (what are users actually doing)?
- When during the day is FlowDesk most used?
- How does feature adoption look over time?

**Note:** You will see an unexplained drop in DAU around November 2023. We'll flag it here and investigate it properly in Notebook 3.

In [ ]:
# Daily Active Users (DAU) — distinct users per day
events['event_date'] = events['event_ts'].dt.date
dau = (
    events
    .groupby('event_date')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'dau'})
)
dau['event_date'] = pd.to_datetime(dau['event_date'])
dau = dau.sort_values('event_date')

# 7-day rolling average for smoother trend
dau['dau_7d_avg'] = dau['dau'].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(dau['event_date'], dau['dau'], alpha=0.15, color='#2563eb')
ax.plot(dau['event_date'], dau['dau'], lw=0.8, color='#93c5fd', alpha=0.7, label='Daily DAU')
ax.plot(dau['event_date'], dau['dau_7d_avg'], lw=2.2, color='#2563eb', label='7-Day Rolling Avg')

# Mark the suspicious drop
bug_start = pd.Timestamp('2023-11-03')
ax.axvline(bug_start, color='#dc2626', lw=1.8, ls='--', alpha=0.85)
ax.text(bug_start + pd.Timedelta(days=2), dau['dau'].max() * 0.92,
        'Investigate?\n(DAU drop ~Nov 3)',
        fontsize=9, color='#dc2626', fontweight='bold')

ax.set_xlabel('Date')
ax.set_ylabel('Daily Active Users')
ax.set_title('FlowDesk Daily Active Users (DAU)\nRaw + 7-Day Rolling Average', fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

print(f"\nPeak DAU:    {dau['dau'].max():,}  on {dau.loc[dau['dau'].idxmax(), 'event_date'].date()}")
print(f"Avg DAU:     {dau['dau'].mean():,.0f}")
print(f"Min DAU:     {dau['dau'].min():,}  on {dau.loc[dau['dau'].idxmin(), 'event_date'].date()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Event type distribution ---
event_counts = events['event_type'].value_counts().sort_values()
colors_ev = sns.color_palette('tab10', len(event_counts))
bars_ev = axes[0].barh(event_counts.index, event_counts.values, color=colors_ev)
for bar, val in zip(bars_ev, event_counts.values):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)
axes[0].set_xlabel('Total Events')
axes[0].set_title('Event Type Distribution\n(All Time)', fontsize=12, fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# --- Hourly activity pattern ---
events['hour'] = events['event_ts'].dt.hour
hourly = events.groupby('hour').size().reset_index(name='events')
axes[1].fill_between(hourly['hour'], hourly['events'], alpha=0.2, color='#10b981')
axes[1].plot(hourly['hour'], hourly['events'], lw=2.2, color='#10b981', marker='o', markersize=4)
axes[1].set_xlabel('Hour of Day (UTC)')
axes[1].set_ylabel('Total Events')
axes[1].set_title('Hourly Activity Pattern\n(When are users most active?)', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(0, 24, 2))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[1].grid(axis='y', alpha=0.25)
peak_hour = hourly.loc[hourly['events'].idxmax(), 'hour']
axes[1].axvline(peak_hour, color='#dc2626', lw=1.5, ls='--', alpha=0.7)
axes[1].text(peak_hour + 0.3, hourly['events'].max() * 0.95,
             f'Peak: {peak_hour}:00', fontsize=9, color='#dc2626')

plt.suptitle('Product Usage Patterns — Events & Time', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature adoption heatmap: feature × week_number
feature_usage['week_start'] = pd.to_datetime(feature_usage['week_start'])
feature_usage['week_num']   = feature_usage['week_start'].dt.isocalendar().week.astype(int)
feature_usage['year']       = feature_usage['week_start'].dt.year

# Aggregate: avg usage per feature per month (aggregating weeks → months for readability)
feature_usage['month'] = feature_usage['week_start'].dt.to_period('M').astype(str)
feat_heatmap = (
    feature_usage
    .groupby(['feature', 'month'])['usage_count']
    .mean()
    .unstack(fill_value=0)
)

# Keep last 18 months for readability
feat_heatmap = feat_heatmap.iloc[:, -18:]

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    feat_heatmap,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.4,
    annot=True,
    fmt='.1f',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Avg Usage Count per User'},
)
ax.set_title('Feature Adoption Over Time\n(Avg Usage Count per User per Month)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Feature')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Section 5 — Churn Overview

Churn is the most important metric for a SaaS business. If you can't retain users, growth is a leaky bucket — any new acquisition immediately drains away.

Here we look at:
- Overall churn rate
- Churn differences across plan tiers
- Churn differences by acquisition channel
- How fast users churn (days to churn by cohort)

In [ ]:
overall_churn = users['is_churned'].mean() * 100
print(f"Overall Churn Rate: {overall_churn:.1f}%")
print(f"Churned Users:      {int(users['is_churned'].sum()):,} of {len(users):,}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Churn rate by plan ---
churn_by_plan = (
    users
    .groupby('plan')['is_churned']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'users'})
    .assign(churn_rate=lambda d: d['churn_rate'] * 100)
)
plan_order_ch = [p for p in ['free', 'pro', 'business', 'enterprise'] if p in churn_by_plan.index]
churn_by_plan = churn_by_plan.reindex(plan_order_ch)

bar_colors_ch = [plan_colors.get(p, '#888') for p in churn_by_plan.index]
bars_ch = axes[0].bar(churn_by_plan.index, churn_by_plan['churn_rate'],
                      color=bar_colors_ch, width=0.55, zorder=3)
for bar, val in zip(bars_ch, churn_by_plan['churn_rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].axhline(overall_churn, color='#6b7280', ls='--', lw=1.5, label=f'Overall avg: {overall_churn:.1f}%')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Plan Tier', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3, zorder=0)

# --- Churn rate by acquisition channel ---
churn_by_channel = (
    users
    .groupby('acquisition_channel')['is_churned']
    .mean()
    .mul(100)
    .sort_values(ascending=True)
)
colors_ch2 = sns.color_palette('RdYlGn_r', len(churn_by_channel))
bars_ch2 = axes[1].barh(churn_by_channel.index, churn_by_channel.values, color=colors_ch2)
for bar, val in zip(bars_ch2, churn_by_channel.values):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
axes[1].axvline(overall_churn, color='#6b7280', ls='--', lw=1.5)
axes[1].set_xlabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Acquisition Channel', fontsize=12, fontweight='bold')

plt.suptitle('FlowDesk Churn Analysis', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Median days-to-churn by cohort_month
churned_users_df = users[users['is_churned'] == 1].copy()
churned_users_df['churn_date']  = pd.to_datetime(churned_users_df['churn_date'])
churned_users_df['days_to_churn'] = (
    churned_users_df['churn_date'] - churned_users_df['signup_date']
).dt.days

median_days = (
    churned_users_df
    .groupby('cohort_month')['days_to_churn']
    .median()
    .reset_index()
    .rename(columns={'days_to_churn': 'median_days_to_churn'})
)
median_days['cohort_dt'] = pd.to_datetime(median_days['cohort_month'] + '-01')
median_days = median_days.sort_values('cohort_dt')

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(median_days['cohort_dt'], median_days['median_days_to_churn'],
                alpha=0.15, color='#f97316')
ax.plot(median_days['cohort_dt'], median_days['median_days_to_churn'],
        lw=2.2, color='#f97316', marker='o', markersize=4)
ax.set_xlabel('Cohort Month')
ax.set_ylabel('Median Days to Churn')
ax.set_title('Median Days-to-Churn by Cohort Month\n'
             '(Higher = users are surviving longer before churning)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Business Insight:**

> Paid users churn at significantly lower rates than free users. The key causality question is: **does upgrading cause retention improvement, or do retained users simply have a higher propensity to upgrade?**
>
> If it's the former, conversion campaigns have a double benefit (revenue + retention). If it's the latter, focusing only on conversion without improving product stickiness will see diminishing returns.
>
> We'll partially answer this in **Notebook 4** using the A/B experiments, and in **Notebook 2** through cohort-level retention analysis.

---
## Section 6 — Key Findings Summary

The table below summarizes the five most actionable findings from this EDA. This is the format used when presenting to a PM or VP — findings tied directly to decisions, not just observations.

| # | Question | Finding | Implication |
|---|----------|---------|-------------|
| 1 | **Who are our users?** | ~60% are on the free plan; enterprise/business users are a small fraction of the user base but the entire revenue base. | Growth strategy should split into two tracks: (a) convert free → paid, (b) expand within existing paid workspaces. |
| 2 | **Do mobile users activate?** | Mobile users activate at ~20+ pp lower rate than desktop users. This represents thousands of lost activations per cohort. | The mobile onboarding experience needs a dedicated audit. Hypothesis: mobile users face friction in the task/project creation flow. |
| 3 | **Is the product growing?** | DAU shows a healthy upward trend overall, but there is a sharp unexplained drop around November 3, 2023 that requires investigation. | Engineering and data need to jointly root-cause the November drop. Could be a bug, a deployment, or a data pipeline issue. |
| 4 | **Which features drive engagement?** | Tasks and page views dominate event volume. Automations and integrations are used far less frequently, but tend to appear in high-retention users. | Promoting automation features during onboarding could improve retention (to be validated with experimentation). |
| 5 | **Who churns and when?** | Enterprise churns at the lowest rate; free plan at the highest. Newer cohorts survive longer before churning than older cohorts. | Product quality improvements since 2023 appear to be working. Quantify this improvement in Notebook 2 (Cohort Analysis). |

---
### Next Steps

- **Notebook 2:** Cohort & Retention Analysis — quantify whether newer cohorts truly retain better and estimate LTV improvement
- **Notebook 3:** Funnel Analysis — dissect the mobile activation gap and identify exactly where users drop off
- **Notebook 4:** A/B Testing — evaluate the two experiments that shipped in 2023 and determine if they caused the retention improvement

---
*FlowDesk Analytics Portfolio | Siddharth Bokka*